# Phase 3: Training R(2+1)D-18 — Optimized for Qualcomm NPU (<34ms)

**Kaggle settings required:**
- **GPU T4 x2** (Settings → Accelerator → GPU T4 x2)
- Internet ON (to download Kinetics-400 pretrained weights)
- Add the output of Notebook 02 as an **input dataset**

**Architecture:** `r2plus1d_18` pretrained on Kinetics-400, fine-tuned on QEVD

**NPU Optimizations (NEW):**
- Input tensors are now **NDHWC** `(1, 16, 112, 112, 3)` — Qualcomm's native layout
- Normalization is **baked into the model graph** (zero CPU overhead at inference)
- Export wrapper handles layout permutation on-device
- INT8 calibration data is generated for quantized compilation

**Progressive unfreezing schedule:**
- Epochs 1-5: only `fc` + `layer4`
- Epochs 6-10: also `layer3` (LR ×0.3)
- Epochs 11-15: also `layer2` (LR ×0.1)

**After running:** Download `best_r2plus1d_qevd.pth` + `qualcomm_r2plus1d.onnx` + `calibration_data/` for the Export notebook.

In [ ]:
import os, json, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')  # headless-safe
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# =========================================================
# CONFIGURATION — update YOUR-PREPROCESSED-DATASET
# =========================================================
TENSORS_ROOT      = '/kaggle/input/notebooks/dadhichisarkershayon/preprocessing-2/preprocessed_tensors'
MANIFEST_PATH     = os.path.join(TENSORS_ROOT, 'manifest.jsonl')
CLASS_LABELS_PATH = os.path.join(TENSORS_ROOT, 'class_labels.json')
CLASS_MAP_PATH    = os.path.join(TENSORS_ROOT, 'class_map.json')

SAVE_DIR          = '/kaggle/working'
BEST_PATH         = os.path.join(SAVE_DIR, 'best_r2plus1d_qevd.pth')
LATEST_PATH       = os.path.join(SAVE_DIR, 'latest_checkpoint.pth')

EPOCHS            = 15
BATCH_SIZE        = 8     # safe for T4 16 GB; use 6 if you get OOM
INITIAL_LR        = 3e-4
WEIGHT_DECAY      = 1e-2
LABEL_SMOOTHING   = 0.1
GRAD_CLIP         = 1.0
NUM_WORKERS       = 4     # T4 kernel has 4 CPU cores

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp  = device.type == 'cuda'
print(f'🔥 Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'   AMP: {use_amp}')

# Fail fast if the input dataset path is wrong
assert os.path.isdir(TENSORS_ROOT), (
    f'TENSORS_ROOT not found: {TENSORS_ROOT}\n'
    f'Update TENSORS_ROOT to match your Kaggle input dataset slug.'
)

In [ ]:
# =========================================================
# LOAD CLASS MAPPING + MANIFEST
# =========================================================
with open(CLASS_LABELS_PATH) as f:
    class_labels = json.load(f)
with open(CLASS_MAP_PATH) as f:
    class_map = json.load(f)

num_classes = len(class_labels)
print(f'✅ {num_classes} classes discovered from metadata')

# =========================================================
# CRITICAL FIX: remap tensor paths
# =========================================================
MARKER = 'preprocessed_tensors'

def remap(old_path: str) -> str:
    # Normalize separators for robustness
    old_path = old_path.replace('\\', '/')
    idx = old_path.find(MARKER)
    if idx == -1:
        return old_path
    suffix = old_path[idx + len(MARKER):]   # e.g. /train/class/file.npy
    return TENSORS_ROOT.rstrip('/') + suffix

all_entries: list[dict] = []
with open(MANIFEST_PATH) as f:
    for line in f:
        line = line.strip()
        if not line: continue
        e = json.loads(line)
        if e['label'] not in class_map: continue
        e['tensor_path'] = remap(e['tensor_path'])
        all_entries.append(e)

assert all_entries, 'Manifest is empty!'
first = all_entries[0]['tensor_path']
print(f'✅ First entry remapped: {first}')

train_entries = [e for e in all_entries if e.get('split') == 'train']
val_entries   = [e for e in all_entries if e.get('split') == 'val']

if not val_entries:
    print('⚠️ No split found in manifest - falling back to 85/15 dynamic split')
    
    # 🔥 Robust dynamic split: duplicate 1-sample classes if they exist
    entries_to_split = []
    labels_for_split = []
    counts = Counter([e['label'] for e in all_entries])
    
    for e in all_entries:
        label = e['label']
        if counts[label] == 1:
            entries_to_split.extend([e, e])
            labels_for_split.extend([label, label])
            print(f'🔔 Duplicating rare class during fallback split: {label}')
        else:
            entries_to_split.append(e)
            labels_for_split.append(label)

    from sklearn.model_selection import train_test_split
    train_entries, val_entries = train_test_split(
        entries_to_split, test_size=0.15, random_state=42, stratify=labels_for_split
    )

print(f'📊 Train: {len(train_entries)} | Val: {len(val_entries)}')

In [ ]:
# =========================================================
# DATASET — handles NDHWC disk format
# =========================================================
class QEVDDataset(Dataset):
    """
    Load pre-processed .npy tensors saved in NDHWC format.
    Tensors on disk: (1, 16, 112, 112, 3) float32 in [0,1].
    
    For PyTorch training, we:
      1. Permute NDHWC -> NCTHW for the standard PyTorch video format
      2. Apply competition mean/std normalization
    """
    # Official competition normalization
    MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)   # (C,1,1,1)
    STD  = torch.tensor([0.22803, 0.22145,  0.216989]).view(3, 1, 1, 1)

    def __init__(self, entries, class_map, augment=False):
        self.entries   = entries
        self.class_map = class_map
        self.augment   = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e    = self.entries[idx]
        arr  = np.load(e['tensor_path'])                # (1, 16, 112, 112, 3) NDHWC
        clip = torch.from_numpy(arr).squeeze(0)         # (16, 112, 112, 3)
        
        # Permute NDHWC -> NCTHW for PyTorch: (T, H, W, C) -> (C, T, H, W)
        clip = clip.permute(3, 0, 1, 2)                 # (3, 16, 112, 112)

        # Normalize
        clip = (clip - self.MEAN) / self.STD

        if self.augment:
            # Random horizontal flip
            if torch.rand(1).item() < 0.5:
                clip = torch.flip(clip, [-1])
            # Random temporal shift (±2 frames)
            if torch.rand(1).item() < 0.3:
                s = torch.randint(-2, 3, (1,)).item()
                if s:
                    clip = torch.roll(clip, shifts=s, dims=1)
            # Brightness jitter
            if torch.rand(1).item() < 0.3:
                clip = clip * (1.0 + (torch.rand(1).item() - 0.5) * 0.3)

        label = self.class_map[e['label']]
        return clip, label


print('✅ Dataset class defined (NDHWC disk → NCTHW training)')

# Quick smoke-test
ds_test = QEVDDataset([train_entries[0]], class_map)
clip_test, lbl_test = ds_test[0]
print(f'   clip shape : {clip_test.shape}')   # expect (3, 16, 112, 112)
print(f'   label      : {lbl_test}')

In [ ]:
# =========================================================
# DATALOADERS — weighted sampler to handle class imbalance
# =========================================================
train_ds = QEVDDataset(train_entries, class_map, augment=True)
val_ds   = QEVDDataset(val_entries,   class_map, augment=False)

train_labels      = [class_map[e['label']] for e in train_entries]
class_counts      = np.bincount(train_labels, minlength=num_classes)
class_weights     = 1.0 / np.clip(class_counts, 1, None)
sample_weights    = [class_weights[l] for l in train_labels]

sampler = WeightedRandomSampler(
    torch.DoubleTensor(sample_weights), len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print(f'✅ Train loader: {len(train_loader)} batches')
print(f'✅ Val   loader: {len(val_loader)} batches')

In [ ]:
# =========================================================
# MODEL
# =========================================================
print('🚀 Building R(2+1)D-18 with Kinetics-400 weights...')
model = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
print(f'✅ FC head replaced: 512 → {num_classes}')

# ----- Progressive freeze helper -----
def set_phase(model, phase: int):
    """Freeze everything, then unfreeze progressively."""
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True
    if phase >= 1:
        for p in model.layer4.parameters(): p.requires_grad = True
    if phase >= 2:
        for p in model.layer3.parameters(): p.requires_grad = True
    if phase >= 3:
        for p in model.layer2.parameters(): p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Phase {phase}: trainable {n_train:,} / {n_total:,} ({100*n_train/n_total:.1f}%)')

set_phase(model, 1)
model = model.to(device)

In [ ]:
# =========================================================
# OPTIMISER + SCHEDULER + AMP
# =========================================================
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def make_opt(model, lr):
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )

optimizer  = make_opt(model, INITIAL_LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
# GradScaler only when GPU is available — initialising it on CPU raises an error
scaler     = torch.amp.GradScaler('cuda') if use_amp else None

print('✅ Optimiser ready')
print(f'   AdamW  lr={INITIAL_LR}  wd={WEIGHT_DECAY}')
print(f'   CosineAnnealingLR  T_max={EPOCHS}')
print(f'   GradScaler: {scaler is not None}')

In [ ]:
# =========================================================
# TRAINING LOOP
# =========================================================
best_val_acc = 0.0
history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

print(f'\n🔥 Training {EPOCHS} epochs  |  batch={BATCH_SIZE}  |  AMP={use_amp}')
print('=' * 70)

for epoch in range(EPOCHS):
    t0 = time.time()

    # ---- Phase transitions ----
    if epoch == 5:
        print('\n🔓 Phase 2: unfreezing layer3')
        set_phase(model, 2)
        optimizer = make_opt(model, INITIAL_LR * 0.3)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - epoch, eta_min=1e-6)
        scaler = torch.amp.GradScaler('cuda') if use_amp else None

    if epoch == 10:
        print('\n🔓 Phase 3: unfreezing layer2')
        set_phase(model, 3)
        optimizer = make_opt(model, INITIAL_LR * 0.1)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - epoch, eta_min=1e-7)
        scaler = torch.amp.GradScaler('cuda') if use_amp else None

    # ---- TRAIN ----
    model.train()
    run_loss = run_correct = run_total = 0

    loop = tqdm(train_loader, desc=f'E{epoch+1:02d} train', leave=False)
    for x, y in loop:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with torch.amp.autocast('cuda'):
                out  = model(x)
                loss = criterion(out, y)
            if torch.isnan(loss): continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(x)
            loss = criterion(out, y)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        run_loss    += loss.item() * x.size(0)
        run_correct += out.argmax(1).eq(y).sum().item()
        run_total   += x.size(0)
        loop.set_postfix(loss=f'{run_loss/max(run_total,1):.3f}',
                         acc=f'{100*run_correct/max(run_total,1):.1f}%')

    scheduler.step()
    tr_loss = run_loss / max(run_total, 1)
    tr_acc  = 100.0 * run_correct / max(run_total, 1)

    # ---- VALIDATE ----
    model.eval()
    vl_loss = vl_correct = vl_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast('cuda'):
                    out  = model(x)
                    loss = criterion(out, y)
            else:
                out  = model(x)
                loss = criterion(out, y)
            if not torch.isnan(loss):
                vl_loss += loss.item() * x.size(0)
            vl_correct += out.argmax(1).eq(y).sum().item()
            vl_total   += y.size(0)

    vl_loss = vl_loss / max(vl_total, 1)
    vl_acc  = 100.0 * vl_correct / max(vl_total, 1)

    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)

    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'val_acc': vl_acc,
            'num_classes': num_classes,
            'class_labels': class_labels,
            'class_map': class_map,
        }, BEST_PATH)
        flag = '  🏆 best'

    # Rolling checkpoint (resume in case of timeout)
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'best_val_acc': best_val_acc,
        'num_classes': num_classes,
    }, LATEST_PATH)

    lr = optimizer.param_groups[0]['lr']
    print(f'E{epoch+1:02d}/{EPOCHS} | '
          f'Train {tr_acc:.2f}% loss={tr_loss:.4f} | '
          f'Val {vl_acc:.2f}% loss={vl_loss:.4f} | '
          f'LR={lr:.1e} | {time.time()-t0:.0f}s{flag}')

print(f'\n🎉 Done! Best val acc: {best_val_acc:.2f}%')
print(f'📂 Download: {BEST_PATH}')

In [ ]:
# =========================================================
# TRAINING CURVES
# =========================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(history['train_acc']) + 1)
ax1.plot(ep, history['train_acc'], 'b-o', ms=4, label='Train')
ax1.plot(ep, history['val_acc'],   'r-o', ms=4, label='Val')
ax1.axhline(90, color='g', ls='--', alpha=0.6, label='90% target')
ax1.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, history['train_loss'], 'b-o', ms=4, label='Train')
ax2.plot(ep, history['val_loss'],   'r-o', ms=4, label='Val')
ax2.set(xlabel='Epoch', ylabel='Loss', title='Loss')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()
print('📈 curves saved')

In [ ]:
# =========================================================
# VERIFY BEST MODEL (clean reload)
# =========================================================
verify = r2plus1d_18(weights=None)
verify.fc = nn.Linear(verify.fc.in_features, num_classes)
ckpt = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
verify.load_state_dict(ckpt['model'])
verify = verify.to(device).eval()

correct = total = 0
with torch.no_grad():
    for x, y in tqdm(val_loader, desc='Final val'):
        out = verify(x.to(device))
        correct += out.argmax(1).eq(y.to(device)).sum().item()
        total   += y.size(0)

print(f'\n✅ Final Val Acc: {100*correct/max(total,1):.2f}%')

# Output shape sanity check
dummy = torch.randn(1, 3, 16, 112, 112).to(device)
with torch.no_grad():
    out = verify(dummy)
print(f'✅ Output shape: {out.shape}  (expect (1, {num_classes}))')
print('\n🚀 Proceeding to NPU-optimized export...')

---
## 🚀 NPU-Optimized Export Pipeline

The cells below create a **wrapper model** that:
1. Accepts NDHWC `(1, 16, 112, 112, 3)` input — Qualcomm's native layout
2. Bakes normalization (mean/std) into the graph — zero CPU overhead
3. Permutes to NCTHW internally for the R(2+1)D backbone
4. Generates INT8 calibration data for quantized compilation

In [ ]:
# =========================================================
# NPU EXPORT WRAPPER — baked normalization + layout permute
# =========================================================
class LPCVC_R2Plus1D_Wrapper(nn.Module):
    """
    Production wrapper for Qualcomm NPU deployment.
    
    Input:  (B, T, H, W, C) = (1, 16, 112, 112, 3) float32 [0, 1]
    Output: (B, num_classes) logits
    
    Internally:
      1. Permutes NDHWC → NCTHW
      2. Normalizes with baked mean/std (runs on NPU)
      3. Forwards through R(2+1)D-18 backbone
    """
    def __init__(self, backbone: nn.Module):
        super().__init__()
        # Register mean/std as buffers so they're embedded in the ONNX graph
        # Shape: (1, C, 1, 1, 1) for broadcasting over (B, C, T, H, W)
        self.register_buffer('mean', torch.tensor(
            [0.43216, 0.394666, 0.37645]
        ).view(1, 3, 1, 1, 1))
        self.register_buffer('std', torch.tensor(
            [0.22803, 0.22145, 0.216989]
        ).view(1, 3, 1, 1, 1))
        self.backbone = backbone
    
    def forward(self, x):
        # x: (B, T, H, W, C) float32 [0,1]  — NDHWC
        # Permute to (B, C, T, H, W) — NCTHW for PyTorch conv3d
        x = x.permute(0, 4, 1, 2, 3)
        # Normalize on-device (baked into graph)
        x = (x - self.mean) / self.std
        return self.backbone(x)


print('✅ LPCVC_R2Plus1D_Wrapper defined')
print('   Input:  (1, 16, 112, 112, 3) NDHWC float32 [0,1]')
print('   Output: (1, num_classes) logits')

In [ ]:
# =========================================================
# ONNX EXPORT — NPU-optimized with baked normalization
# =========================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx'], check=True)

ONNX_PATH = os.path.join(SAVE_DIR, 'qualcomm_r2plus1d.onnx')

# 1. Load best weights into a fresh backbone
print('🔄 Loading best weights...')
backbone = r2plus1d_18(weights=None)
backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)
ckpt = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
backbone.load_state_dict(ckpt['model'])

# 2. Wrap with NPU-optimized preprocessing
export_model = LPCVC_R2Plus1D_Wrapper(backbone)
export_model.eval()

# 3. Create NDHWC dummy input — (1, 16, 112, 112, 3)
dummy_input = torch.randn(1, 16, 112, 112, 3)

# 4. Verify forward pass
with torch.no_grad():
    test_out = export_model(dummy_input)
    print(f'✅ Forward pass OK: input {dummy_input.shape} → output {test_out.shape}')

# 5. Export ONNX — legacy path for reliable weight embedding
print(f'🚀 Exporting ONNX to {ONNX_PATH}...')
torch.onnx.export(
    export_model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=14,          # Stable for Qualcomm QNN
    do_constant_folding=True,  # Folds BatchNorm + normalization constants
    input_names=['input'],
    output_names=['output'],
    dynamo=False,              # Force legacy exporter for reliable weight embedding
    keep_initializers_as_inputs=False,
    dynamic_axes=None          # Fixed shapes = faster NPU inference
)

# 6. Verify file size
size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print(f'📏 ONNX Size: {size_mb:.2f} MB')
if size_mb > 100:
    print('✅ SUCCESS! Weights are embedded.')
else:
    print('⚠️ Warning: File may be missing weights. Attempting unification...')
    import onnx
    m = onnx.load(ONNX_PATH)
    unified_path = ONNX_PATH.replace('.onnx', '_unified.onnx')
    onnx.save(m, unified_path, save_as_external_data=False)
    size_mb = os.path.getsize(unified_path) / (1024 * 1024)
    print(f'📏 Unified ONNX Size: {size_mb:.2f} MB')
    ONNX_PATH = unified_path

print(f'\n📂 ONNX file: {ONNX_PATH}')

In [ ]:
# =========================================================
# GENERATE INT8 CALIBRATION DATA
# =========================================================
# Qualcomm AI Hub needs representative samples for quantization.
# We save 100 samples as individual .npy files in NDHWC format.
# These are UN-normalized (raw [0,1]) since the wrapper handles normalization.
# =========================================================
import random

CALIB_DIR = os.path.join(SAVE_DIR, 'calibration_data')
CALIB_COUNT = 100  # 50-100 is recommended by Qualcomm

os.makedirs(CALIB_DIR, exist_ok=True)

# Sample from training entries
random.seed(42)
calib_entries = random.sample(train_entries, min(CALIB_COUNT, len(train_entries)))

print(f'📊 Generating {len(calib_entries)} calibration samples...')

calib_inputs = []
for i, e in enumerate(tqdm(calib_entries, desc='Calibration')):
    arr = np.load(e['tensor_path'])  # (1, 16, 112, 112, 3) NDHWC float32
    calib_inputs.append(arr)

# Stack into a single array: (N, 16, 112, 112, 3)
calib_array = np.concatenate(calib_inputs, axis=0)  # (100, 16, 112, 112, 3)
np.save(os.path.join(CALIB_DIR, 'calibration_inputs.npy'), calib_array)

print(f'✅ Calibration data saved: {calib_array.shape}')
print(f'   Location: {CALIB_DIR}')
print(f'   dtype: {calib_array.dtype}, range: [{calib_array.min():.3f}, {calib_array.max():.3f}]')
print(f'\n📦 Download these files for local AI Hub submission:')
print(f'   1. {ONNX_PATH}')
print(f'   2. {CALIB_DIR}/calibration_inputs.npy')
print(f'   3. {BEST_PATH}')